# 🗺️ Route Analysis

Explore flight routes, popular destinations, and travel patterns.

## Setup

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

df = pd.read_excel('airline_ticket_dataset.xlsx')
df['route'] = df['city1'] + ' → ' + df['city2']
print(f"✅ Loaded {len(df):,} flight records")

✅ Loaded 14,004 flight records


## 1. Most Popular Routes (by Passenger Count)

Which routes do people fly the most?

In [2]:
# Aggregate by route
route_stats = df.groupby('route').agg({
    'passengers': 'sum',
    'nsmiles': 'mean',
    'fare': 'mean'
}).reset_index()

# Top 20 by passengers
top_routes = route_stats.nlargest(20, 'passengers')

fig = px.bar(top_routes, 
             x='passengers', 
             y='route',
             orientation='h',
             title='Top 20 Most Popular Routes (Total Passengers)',
             labels={'passengers': 'Total Passengers', 'route': 'Route'},
             color='passengers',
             color_continuous_scale='Blues',
             hover_data={'fare': ':.2f', 'nsmiles': ':.0f'})

fig.update_layout(height=600, yaxis={'categoryorder':'total ascending'})
fig.update_traces(texttemplate='%{x:,}', textposition='outside')
fig.show()

## 2. Route Distance Distribution

Short-haul vs long-haul flights:

In [3]:
# Categorize by distance
df['distance_category'] = pd.cut(df['nsmiles'], 
                                  bins=[0, 500, 1000, 1500, 3000],
                                  labels=['Short (<500 mi)', 'Medium (500-1000 mi)', 
                                          'Long (1000-1500 mi)', 'Very Long (>1500 mi)'])

dist_counts = df['distance_category'].value_counts().reset_index()
dist_counts.columns = ['Distance Category', 'Number of Routes']

fig = px.pie(dist_counts, 
             values='Number of Routes', 
             names='Distance Category',
             title='Flight Routes by Distance Category',
             hole=0.4,
             color_discrete_sequence=px.colors.sequential.RdBu)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 3. Distance vs Passenger Volume

Do people prefer shorter or longer flights?

In [4]:
fig = px.scatter(route_stats, 
                 x='nsmiles', 
                 y='passengers',
                 size='passengers',
                 color='fare',
                 hover_data=['route'],
                 title='Route Distance vs Passenger Volume',
                 labels={'nsmiles': 'Distance (miles)', 'passengers': 'Total Passengers', 'fare': 'Avg Fare ($)'},
                 color_continuous_scale='Viridis',
                 opacity=0.6)

fig.update_layout(height=600)
fig.show()

## 4. Top Origin Cities (Most Departures)

Which cities are the busiest departure points?

In [5]:
origin_stats = df.groupby('city1').agg({
    'passengers': 'sum',
    'route': 'count',
    'fare': 'mean'
}).reset_index()
origin_stats.columns = ['City', 'Total Passengers', 'Number of Routes', 'Avg Fare']

top_origins = origin_stats.nlargest(15, 'Total Passengers')

fig = px.bar(top_origins, 
             x='City', 
             y='Total Passengers',
             title='Top 15 Origin Cities (Busiest Departure Points)',
             labels={'Total Passengers': 'Total Passengers', 'City': 'Origin City'},
             color='Number of Routes',
             color_continuous_scale='Oranges',
             hover_data={'Avg Fare': ':.2f'})

fig.update_layout(xaxis_tickangle=-45, height=600)
fig.show()

## 5. Top Destination Cities

Where are people flying to?

In [6]:
dest_stats = df.groupby('city2').agg({
    'passengers': 'sum',
    'route': 'count',
    'fare': 'mean'
}).reset_index()
dest_stats.columns = ['City', 'Total Passengers', 'Number of Routes', 'Avg Fare']

top_dests = dest_stats.nlargest(15, 'Total Passengers')

fig = px.bar(top_dests, 
             x='City', 
             y='Total Passengers',
             title='Top 15 Destination Cities (Most Arrivals)',
             labels={'Total Passengers': 'Total Passengers', 'City': 'Destination City'},
             color='Number of Routes',
             color_continuous_scale='Purples',
             hover_data={'Avg Fare': ':.2f'})

fig.update_layout(xaxis_tickangle=-45, height=600)
fig.show()

## 6. Route Network Complexity

How many routes does each city have?

In [ ]:
# Count unique routes per city (as origin or destination)
city_routes_origin = df.groupby('city1')['city2'].nunique().reset_index()
city_routes_origin.columns = ['City', 'Connections as Origin']

city_routes_dest = df.groupby('city2')['city1'].nunique().reset_index()
city_routes_dest.columns = ['City', 'Connections as Destination']

# Merge
city_connections = pd.merge(city_routes_origin, city_routes_dest, on='City', how='outer').fillna(0)
city_connections['Total Connections'] = city_connections['Connections as Origin'] + city_connections['Connections as Destination']

top_connected = city_connections.nlargest(20, 'Total Connections')

fig = go.Figure()
fig.add_trace(go.Bar(name='As Origin', x=top_connected['City'], y=top_connected['Connections as Origin']))
fig.add_trace(go.Bar(name='As Destination', x=top_connected['City'], y=top_connected['Connections as Destination']))

fig.update_layout(
    title='Top 20 Most Connected Cities',
    xaxis_title='City',
    yaxis_title='Number of Unique Connections',
    barmode='stack',
    xaxis_tickangle=-45,
    height=600
)
fig.show()

## 7. Long-Haul vs Short-Haul Popularity

Passenger preference by distance:

In [ ]:
distance_passengers = df.groupby('distance_category')['passengers'].sum().reset_index()

fig = px.bar(distance_passengers, 
             x='distance_category', 
             y='passengers',
             title='Total Passengers by Flight Distance',
             labels={'distance_category': 'Distance Category', 'passengers': 'Total Passengers'},
             color='passengers',
             color_continuous_scale='Blues',
             text='passengers')

fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

## 8. Busiest City Pairs (Bidirectional)

Which city pairs have the most traffic in both directions?

In [ ]:
# Create bidirectional pairs
df['city_pair'] = df.apply(lambda row: ' ↔ '.join(sorted([row['city1'], row['city2']])), axis=1)

pair_stats = df.groupby('city_pair').agg({
    'passengers': 'sum',
    'fare': 'mean',
    'nsmiles': 'mean'
}).reset_index()

top_pairs = pair_stats.nlargest(20, 'passengers')

fig = px.bar(top_pairs, 
             x='passengers', 
             y='city_pair',
             orientation='h',
             title='Top 20 Busiest City Pairs (Combined Both Directions)',
             labels={'passengers': 'Total Passengers', 'city_pair': 'City Pair'},
             color='fare',
             color_continuous_scale='Turbo',
             hover_data={'nsmiles': ':.0f', 'fare': ':.2f'})

fig.update_layout(height=600, yaxis={'categoryorder':'total ascending'})
fig.show()

## 💡 Route Insights Summary

In [ ]:
print("="*60)
print("🗺️ ROUTE ANALYSIS INSIGHTS")
print("="*60)

most_popular = route_stats.nlargest(1, 'passengers').iloc[0]
print(f"\n✈️ Most Popular Route:")
print(f"   • {most_popular['route']}")
print(f"   • Total passengers: {most_popular['passengers']:,.0f}")
print(f"   • Average fare: ${most_popular['fare']:.2f}")

busiest_origin = origin_stats.nlargest(1, 'Total Passengers').iloc[0]
busiest_dest = dest_stats.nlargest(1, 'Total Passengers').iloc[0]
print(f"\n🏙️ Busiest Cities:")
print(f"   • Origin: {busiest_origin['City']} ({busiest_origin['Total Passengers']:,.0f} passengers)")
print(f"   • Destination: {busiest_dest['City']} ({busiest_dest['Total Passengers']:,.0f} passengers)")

print(f"\n📏 Distance Analysis:")
print(f"   • Average route distance: {df['nsmiles'].mean():.0f} miles")
print(f"   • Shortest route: {df['nsmiles'].min():.0f} miles")
print(f"   • Longest route: {df['nsmiles'].max():.0f} miles")

most_connected = city_connections.nlargest(1, 'Total Connections').iloc[0]
print(f"\n🌐 Most Connected City:")
print(f"   • {most_connected['City']}")
print(f"   • Total connections: {int(most_connected['Total Connections'])}")

print("\n" + "="*60)